In [ ]:
!pip install -q groq gradio

In [ ]:
import os
import gradio as gr
from groq import Groq
from getpass import getpass

# Enter your Groq key (or use Colab Secrets: from google.colab import userdata)
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")
client = Groq(api_key=os.environ["GROQ_API_KEY"])

MODEL = "openai/gpt-oss-120b"  # alt: "llama-3.1-8b-instant" (faster/cheaper)
MAX_TURNS = 20  # how many past messages to remember

SYSTEM_PROMPT = (
    "You are a friendly, helpful assistant. Remember what the user "
    "told you earlier in the conversation and use it to give relevant, "
    "personalized answers. Keep replies clear and concise."
)

def chat(message, history):
    # Build the message list: system prompt + past turns + new message
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for turn in history[-MAX_TURNS:]:
        messages.append({"role": turn["role"], "content": turn["content"]})
    messages.append({"role": "user", "content": message})

    # Stream the reply for a real-time feel
    stream = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.7,
        max_tokens=1024,
        stream=True,
    )
    reply = ""
    for chunk in stream:
        reply += chunk.choices[0].delta.content or ""
        yield reply

demo = gr.ChatInterface(
    fn=chat,
    type="messages",
    title="💬 Context-Aware AI Assistant",
    description="Chat in real time. I remember everything we've said in this session.",
    examples=["Hi, my name is Sam and I love cooking", "What was my name again?"],
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2ea94b669061265bfe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
